<a href="https://colab.research.google.com/github/smshozab/AG-AquaSense/blob/main/AAIn7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview

This notebook integrates all modules into a **complete AquaSense-Agent system**.

### What this notebook does
- Implements intent routing:
  - DIAGNOSIS, WATER_QUALITY, INFORMATION, GREETING
- Runs diagnosis pipeline:
  - P_text + P_sensor + P_cnn → fusion
- Performs knowledge retrieval using RAG
- Generates final advisory output:
  - primary diagnosis
  - top 3 differentials
  - sensor interpretation
  - recommended actions
- Measures latency across components

### Key outputs
- End-to-end advisory responses
- Latency table (mean, P50, P95)
- Structured prompt format

### Usage
- Final system demonstration and evaluation

### Note
- CNN is currently simulated; full system works end-to-end

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install faiss-cpu rank_bm25 sentence-transformers

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/aquasense")

TEXT_PATH = BASE_DIR / "notebook3_outputs" / "p_text_predictions.csv"
SENSOR_PATH = BASE_DIR / "notebook4_outputs" / "p_sensor_predictions.csv"
CNN_PATH = BASE_DIR / "notebook5_outputs" / "p_cnn_simulated.csv"

LATENCY_OUTPUT_DIR = BASE_DIR / "notebook7_outputs"
LATENCY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DISEASE_CLASSES = [
    "Bacterial Red Disease",
    "Aeromoniasis",
    "Bacterial Gill Disease",
    "Saprolegniasis",
    "Healthy Fish",
    "Parasitic Diseases",
    "White Tail Disease"
]

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
p_text_df = pd.read_csv(TEXT_PATH)
p_sensor_df = pd.read_csv(SENSOR_PATH)
p_cnn_df = pd.read_csv(CNN_PATH)

print("Loaded shapes:")
print("Text  :", p_text_df.shape)
print("Sensor:", p_sensor_df.shape)
print("CNN   :", p_cnn_df.shape)

display(p_text_df.head())
display(p_sensor_df.head())
display(p_cnn_df.head())

Loaded shapes:
Text  : (3500, 10)
Sensor: (2205, 21)
CNN   : (2205, 9)


,text,true_label,pred_label,P_text_Bacterial Red Disease,P_text_Aeromoniasis,P_text_Bacterial Gill Disease,P_text_Saprolegniasis,P_text_Healthy Fish,P_text_Parasitic Diseases,P_text_White Tail Disease
0,pls help are red spots near fins also slow one...,Bacterial Red Disease,Bacterial Red Disease,1.000008,0.000010,0.000010,0.000011,0.000011,0.000011,0.000010
1,ulcers and reed lesions on body many fish affe...,Bacterial Red Disease,Bacterial Red Disease,0.999867,0.000148,0.000010,0.000015,0.000010,0.000010,0.000010
2,can you check fish fish has rd patches on bood...,Bacterial Red Disease,Bacterial Red Disease,0.995325,0.000022,0.000010,0.004681,0.000010,0.000012,0.000010
3,WHAT DISEASE IS THIS THIS FISH HAS HAS RD PATC...,Bacterial Red Disease,Bacterial Red Disease,0.979547,0.002005,0.000032,0.016084,0.000113,0.001871,0.000418
4,FISSH LOOKING BAD THERE ARE RED FINS PLUS SWIM...,Bacterial Red Disease,Bacterial Red Disease,0.998806,0.000579,0.000011,0.000532,0.000039,0.000089,0.000013


,timestamp,tank_id,do,ph,temp,tan,turbidity,a_do,a_ph,a_temp,...,a_turb,disease_label,sensor_pred_label,P_sensor_Aeromoniasis,P_sensor_Bacterial Gill Disease,P_sensor_Bacterial Red Disease,P_sensor_Healthy Fish,P_sensor_Parasitic Diseases,P_sensor_Saprolegniasis,P_sensor_White Tail Disease
0,2025-01-13 01:31:00,tank_1,6.808621,7.486847,27.449030,0.465207,10.429919,1.344428,1.009459,-0.220508,...,0.062595,White Tail Disease,White Tail Disease,0.003975,0.001033,0.009600,0.000373,0.003276,0.004669,0.977075
1,2025-01-11 18:58:00,tank_1,6.059563,7.363987,26.667226,0.388589,8.967176,-1.243605,-0.399945,-1.475268,...,-1.071868,Saprolegniasis,Saprolegniasis,0.000038,0.000063,0.000043,0.002044,0.000167,0.997610,0.000035
2,2025-01-05 09:15:00,tank_1,6.964514,7.401811,28.224975,0.342951,9.833359,0.784121,0.011361,1.283974,...,0.535652,Healthy Fish,Healthy Fish,0.006801,0.000425,0.000193,0.992414,0.000093,0.000028,0.000045
3,2025-01-08 04:15:00,tank_1,5.835228,7.501221,28.845101,0.679509,13.148290,-1.224627,1.160706,1.724881,...,2.091577,Aeromoniasis,Aeromoniasis,0.993668,0.004339,0.000225,0.000018,0.001407,0.000045,0.000297
4,2025-01-14 14:18:00,tank_1,6.105284,7.277716,27.294417,0.389847,9.364911,-0.906522,-1.417915,-0.364121,...,-0.833801,Parasitic Diseases,Parasitic Diseases,0.000061,0.000962,0.000728,0.002652,0.991236,0.004171,0.000191


,fusion_id,true_label,P_cnn_Bacterial Red Disease,P_cnn_Aeromoniasis,P_cnn_Bacterial Gill Disease,P_cnn_Saprolegniasis,P_cnn_Healthy Fish,P_cnn_Parasitic Diseases,P_cnn_White Tail Disease
0,0,Bacterial Red Disease,0.466505,0.061200,0.119729,0.097246,0.013133,0.136048,0.106139
1,1,Bacterial Red Disease,0.501105,0.019121,0.067220,0.055341,0.138319,0.096097,0.122797
2,2,Bacterial Red Disease,0.496036,0.037388,0.091247,0.010500,0.136171,0.103929,0.124729
3,3,Bacterial Red Disease,0.466412,0.154734,0.142368,0.124078,0.031026,0.074398,0.006983
4,4,Bacterial Red Disease,0.433557,0.108647,0.118463,0.153893,0.051826,0.058926,0.074688


In [ ]:
p_text_df = p_text_df.reset_index(drop=True).copy()
p_sensor_df = p_sensor_df.reset_index(drop=True).copy()
p_cnn_df = p_cnn_df.reset_index(drop=True).copy()

n = min(len(p_text_df), len(p_sensor_df), len(p_cnn_df))

p_text_df = p_text_df.iloc[:n].copy()
p_sensor_df = p_sensor_df.iloc[:n].copy()
p_cnn_df = p_cnn_df.iloc[:n].copy()

p_text_df["fusion_id"] = np.arange(n)
p_sensor_df["fusion_id"] = np.arange(n)
p_cnn_df["fusion_id"] = np.arange(n)

print("Aligned row count:", n)

Aligned row count: 2205


In [ ]:
knowledge_data = [
    {"type": "qa", "disease": "Bacterial Gill Disease",
     "content": "Fish gasping near the surface often indicates low dissolved oxygen or gill disease. Increase aeration and check ammonia."},

    {"type": "qa", "disease": "Aeromoniasis",
     "content": "Ulcers, red lesions, poor appetite, and lethargy can indicate Aeromoniasis. Improve water quality and isolate affected fish."},

    {"type": "qa", "disease": "Saprolegniasis",
     "content": "Cotton-like fungal growth on skin or fins is commonly associated with saprolegniasis."},

    {"type": "qa", "disease": "Parasitic Diseases",
     "content": "Rubbing against tank surfaces, flashing, and irritation are common signs of parasitic disease."},

    {"type": "qa", "disease": "White Tail Disease",
     "content": "White discoloration in the tail region with weak swimming suggests white tail disease."},

    {"type": "qa", "disease": "Bacterial Red Disease",
     "content": "Red patches, hemorrhagic spots, and inflamed skin suggest bacterial red disease."},

    {"type": "qa", "disease": "Healthy Fish",
     "content": "Healthy fish are active, eat regularly, show no lesions, and maintain normal swimming behavior."},

    {"type": "literature", "disease": "Bacterial Gill Disease",
     "content": "Bacterial gill disease is associated with respiratory distress, low dissolved oxygen, elevated ammonia, and poor water exchange."},

    {"type": "literature", "disease": "Aeromoniasis",
     "content": "Aeromoniasis often appears under warm, high-stress, high-organic-load conditions and presents with ulcerative lesions."},

    {"type": "literature", "disease": "Saprolegniasis",
     "content": "Fungal infections such as saprolegniasis are more likely in stressed fish and damaged skin tissue."},

    {"type": "case", "disease": "Bacterial Gill Disease",
     "content": "Case record: fish showed rapid breathing, low DO anomaly, elevated TAN anomaly, and gill inflammation was later confirmed."},

    {"type": "case", "disease": "Aeromoniasis",
     "content": "Case record: fish displayed ulcers, weak feeding, high ammonia, and warm water stress. Aeromoniasis was confirmed."},

    {"type": "case", "disease": "Parasitic Diseases",
     "content": "Case record: repeated flashing behavior, rubbing on tank wall, skin irritation, and parasite presence under microscopy."},
]

kb_df = pd.DataFrame(knowledge_data)
display(kb_df.head())

,type,disease,content
0,qa,Bacterial Gill Disease,Fish gasping near the surface often indicates ...
1,qa,Aeromoniasis,"Ulcers, red lesions, poor appetite, and lethar..."
2,qa,Saprolegniasis,Cotton-like fungal growth on skin or fins is c...
3,qa,Parasitic Diseases,"Rubbing against tank surfaces, flashing, and i..."
4,qa,White Tail Disease,White discoloration in the tail region with we...


In [ ]:
def chunk_text(text, chunk_size=40):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

chunks = []
for _, row in kb_df.iterrows():
    text_chunks = chunk_text(row["content"])
    for c in text_chunks:
        chunks.append({
            "text": c,
            "disease": row["disease"],
            "type": row["type"]
        })

chunk_df = pd.DataFrame(chunks)
display(chunk_df.head())

,text,disease,type
0,Fish gasping near the surface often indicates ...,Bacterial Gill Disease,qa
1,"Ulcers, red lesions, poor appetite, and lethar...",Aeromoniasis,qa
2,Cotton-like fungal growth on skin or fins is c...,Saprolegniasis,qa
3,"Rubbing against tank surfaces, flashing, and i...",Parasitic Diseases,qa
4,White discoloration in the tail region with we...,White Tail Disease,qa


In [ ]:
tokenized_corpus = [doc.split() for doc in chunk_df["text"]]
bm25 = BM25Okapi(tokenized_corpus)

chunk_embeddings = embedding_model.encode(chunk_df["text"].tolist(), show_progress_bar=True)
faiss_index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
faiss_index.add(chunk_embeddings)

print("BM25 and FAISS indexes built.")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

BM25 and FAISS indexes built.


In [ ]:
def bm25_search(query, top_k=5):
    tokenized_query = query.split()
    scores = bm25.get_scores(tokenized_query)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return top_idx

def faiss_search(query, top_k=5):
    q_emb = embedding_model.encode([query])
    distances, indices = faiss_index.search(q_emb, top_k)
    return indices[0]

def rrf_merge(bm25_idx, faiss_idx, k=60):
    scores = {}

    for rank, idx in enumerate(bm25_idx):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    for rank, idx in enumerate(faiss_idx):
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in ranked]

def retrieve_top_chunks(query, top_k=3):
    bm_idx = bm25_search(query, top_k=5)
    fa_idx = faiss_search(query, top_k=5)
    final_idx = rrf_merge(bm_idx, fa_idx)
    return chunk_df.iloc[final_idx[:top_k]].copy()

In [ ]:
def classify_intent(user_query):
    q = user_query.lower()

    greeting_keywords = ["hello", "hi", "hey", "good morning", "good evening"]
    water_keywords = ["do", "dissolved oxygen", "ph", "temperature", "ammonia", "tan", "turbidity", "water quality"]
    diagnosis_keywords = ["disease", "symptom", "lesion", "gasping", "ulcer", "tail", "fungus", "spots", "diagnose", "what is wrong"]
    info_keywords = ["what is", "tell me about", "information", "explain", "treatment", "how to manage"]

    if any(k in q for k in greeting_keywords):
        return "GREETING"
    elif any(k in q for k in water_keywords) and not any(k in q for k in diagnosis_keywords):
        return "WATER_QUALITY"
    elif any(k in q for k in diagnosis_keywords):
        return "DIAGNOSIS"
    elif any(k in q for k in info_keywords):
        return "INFORMATION"
    else:
        return "INFORMATION"

In [ ]:
def normalize_probs(prob_dict):
    total = sum(prob_dict.values())
    if total == 0:
        return {k: 1.0 / len(prob_dict) for k in prob_dict}
    return {k: v / total for k, v in prob_dict.items()}

def get_text_probs_from_query(query):
    q = query.lower()

    probs = {cls: 0.02 for cls in DISEASE_CLASSES}

    if any(k in q for k in ["gasping", "breathing fast", "surface", "gills"]):
        probs["Bacterial Gill Disease"] += 0.70
        probs["Aeromoniasis"] += 0.10

    if any(k in q for k in ["ulcer", "lesion", "red spots", "red patch", "hemorrhage"]):
        probs["Bacterial Red Disease"] += 0.45
        probs["Aeromoniasis"] += 0.35

    if any(k in q for k in ["cotton", "fungus", "white growth", "fuzzy"]):
        probs["Saprolegniasis"] += 0.75

    if any(k in q for k in ["rubbing", "flashing", "itching", "irritated"]):
        probs["Parasitic Diseases"] += 0.75

    if any(k in q for k in ["white tail", "tail white", "tail pale"]):
        probs["White Tail Disease"] += 0.80

    if any(k in q for k in ["healthy", "normal", "active", "eating fine"]):
        probs["Healthy Fish"] += 0.75

    return normalize_probs(probs)

def get_sensor_probs_from_sample(sample_row):
    prob_cols = [f"P_sensor_{cls}" for cls in DISEASE_CLASSES]
    probs = {cls: float(sample_row[f"P_sensor_{cls}"]) for cls in DISEASE_CLASSES}
    return normalize_probs(probs)

def get_cnn_probs_from_sample(sample_row):
    prob_cols = [f"P_cnn_{cls}" for cls in DISEASE_CLASSES]
    probs = {cls: float(sample_row[f"P_cnn_{cls}"]) for cls in DISEASE_CLASSES}
    return normalize_probs(probs)

In [ ]:
macro_f1_cnn = 0.88
macro_f1_text = 0.90
macro_f1_sensor = 0.95

weight_sum = macro_f1_cnn + macro_f1_text + macro_f1_sensor
w_cnn = macro_f1_cnn / weight_sum
w_text = macro_f1_text / weight_sum
w_sensor = macro_f1_sensor / weight_sum

print("Fusion weights:")
print("w_cnn   =", w_cnn)
print("w_text  =", w_text)
print("w_sensor=", w_sensor)

Fusion weights:
w_cnn   = 0.32234432234432236
w_text  = 0.32967032967032966
w_sensor= 0.34798534798534797


In [ ]:
def fuse_modalities(cnn_probs, text_probs, sensor_probs):
    fused = {}
    for cls in DISEASE_CLASSES:
        fused[cls] = (
            w_cnn * cnn_probs.get(cls, 0.0) +
            w_text * text_probs.get(cls, 0.0) +
            w_sensor * sensor_probs.get(cls, 0.0)
        )
    return normalize_probs(fused)

def top_k_predictions(prob_dict, k=3):
    return sorted(prob_dict.items(), key=lambda x: x[1], reverse=True)[:k]

In [ ]:
def interpret_sensor_state(sample_row):
    notes = []

    if "a_do" in sample_row and sample_row["a_do"] < -2:
        notes.append("Dissolved oxygen is significantly below recent baseline.")
    elif "do" in sample_row and sample_row["do"] < 6:
        notes.append("Dissolved oxygen appears low for stable fish health.")

    if "a_tan" in sample_row and sample_row["a_tan"] > 2:
        notes.append("Ammonia anomaly is elevated and may be stressing the fish.")
    elif "tan" in sample_row and sample_row["tan"] > 0.45:
        notes.append("Ammonia level is above the healthier range.")

    if "a_temp" in sample_row and abs(sample_row["a_temp"]) > 2:
        notes.append("Temperature is showing abnormal deviation from recent history.")

    if "a_turb" in sample_row and sample_row["a_turb"] > 2:
        notes.append("Turbidity is unusually high and may indicate degraded water conditions.")

    if "a_ph" in sample_row and abs(sample_row["a_ph"]) > 2:
        notes.append("pH is outside its recent normal pattern.")

    if not notes:
        notes.append("Sensor readings do not show major acute anomalies at this moment.")

    return notes

In [ ]:
def generate_advisory(user_query, fused_probs, sample_row, retrieved_chunks):
    top3 = top_k_predictions(fused_probs, k=3)
    primary = top3[0]
    sensor_notes = interpret_sensor_state(sample_row)

    actions = []

    if primary[0] == "Bacterial Gill Disease":
        actions = [
            "Increase aeration immediately.",
            "Check ammonia and reduce organic waste load.",
            "Inspect gill condition and isolate severely affected fish if possible."
        ]
    elif primary[0] == "Aeromoniasis":
        actions = [
            "Improve water quality and reduce stress conditions.",
            "Isolate fish showing ulcerative lesions.",
            "Review feeding, waste load, and hygiene conditions."
        ]
    elif primary[0] == "Saprolegniasis":
        actions = [
            "Inspect for fungal growth and skin damage.",
            "Reduce stressors and improve sanitation.",
            "Separate visibly affected fish where possible."
        ]
    elif primary[0] == "Parasitic Diseases":
        actions = [
            "Inspect fish for external irritation and abnormal behavior.",
            "Check stocking density and tank cleanliness.",
            "Consider parasite-specific follow-up inspection."
        ]
    elif primary[0] == "White Tail Disease":
        actions = [
            "Monitor tail discoloration and swimming weakness closely.",
            "Reduce environmental stress and maintain stable water quality.",
            "Separate suspicious cases for closer observation."
        ]
    elif primary[0] == "Bacterial Red Disease":
        actions = [
            "Inspect red lesions or hemorrhagic patches carefully.",
            "Improve water quality and reduce stress immediately.",
            "Monitor for worsening skin damage or secondary infections."
        ]
    else:
        actions = [
            "Continue routine monitoring.",
            "Maintain stable water quality and feeding patterns.",
            "Reassess if symptoms or water conditions change."
        ]

    retrieved_text = retrieved_chunks["text"].tolist()

    advisory = {
        "user_query": user_query,
        "primary_diagnosis": primary[0],
        "primary_confidence": round(primary[1], 4),
        "top_3_differentials": [(cls, round(score, 4)) for cls, score in top3],
        "sensor_interpretation": sensor_notes,
        "retrieved_knowledge": retrieved_text,
        "recommended_actions": actions
    }

    return advisory

In [ ]:
def run_aquasense_agent(user_query, sample_idx=0):
    latency = {}

    t0 = time.perf_counter()
    intent = classify_intent(user_query)
    latency["intent_routing_ms"] = (time.perf_counter() - t0) * 1000

    sample_idx = sample_idx % len(p_sensor_df)
    sensor_row = p_sensor_df.iloc[sample_idx]
    cnn_row = p_cnn_df.iloc[sample_idx]

    if intent == "GREETING":
        response = {
            "intent": intent,
            "message": "Hello. I can help with fish disease diagnosis, water quality interpretation, and aquaculture information."
        }
        latency["end_to_end_ms"] = sum(latency.values())
        return response, latency

    if intent == "WATER_QUALITY":
        t1 = time.perf_counter()
        sensor_notes = interpret_sensor_state(sensor_row)
        latency["sensor_processing_ms"] = (time.perf_counter() - t1) * 1000

        t2 = time.perf_counter()
        chunks = retrieve_top_chunks(user_query, top_k=3)
        latency["rag_retrieval_ms"] = (time.perf_counter() - t2) * 1000

        response = {
            "intent": intent,
            "water_quality_notes": sensor_notes,
            "retrieved_knowledge": chunks["text"].tolist(),
            "message": "Water quality analysis completed."
        }
        latency["end_to_end_ms"] = sum(latency.values())
        return response, latency

    if intent == "INFORMATION":
        t2 = time.perf_counter()
        chunks = retrieve_top_chunks(user_query, top_k=3)
        latency["rag_retrieval_ms"] = (time.perf_counter() - t2) * 1000

        response = {
            "intent": intent,
            "retrieved_knowledge": chunks["text"].tolist(),
            "message": "Information retrieval completed."
        }
        latency["end_to_end_ms"] = sum(latency.values())
        return response, latency

    if intent == "DIAGNOSIS":
        t_text = time.perf_counter()
        text_probs = get_text_probs_from_query(user_query)
        latency["text_inference_ms"] = (time.perf_counter() - t_text) * 1000

        t_sensor = time.perf_counter()
        sensor_probs = get_sensor_probs_from_sample(sensor_row)
        latency["sensor_inference_ms"] = (time.perf_counter() - t_sensor) * 1000

        t_cnn = time.perf_counter()
        cnn_probs = get_cnn_probs_from_sample(cnn_row)
        latency["cnn_inference_ms"] = (time.perf_counter() - t_cnn) * 1000

        t_fusion = time.perf_counter()
        fused_probs = fuse_modalities(cnn_probs, text_probs, sensor_probs)
        latency["fusion_ms"] = (time.perf_counter() - t_fusion) * 1000

        t_rag = time.perf_counter()
        primary = top_k_predictions(fused_probs, 1)[0][0]
        rag_query = f"{user_query} {primary}"
        retrieved_chunks = retrieve_top_chunks(rag_query, top_k=3)
        latency["rag_retrieval_ms"] = (time.perf_counter() - t_rag) * 1000

        t_gen = time.perf_counter()
        advisory = generate_advisory(user_query, fused_probs, sensor_row, retrieved_chunks)
        latency["advisory_generation_ms"] = (time.perf_counter() - t_gen) * 1000

        advisory["intent"] = intent
        latency["end_to_end_ms"] = sum(latency.values())
        return advisory, latency

In [ ]:
def print_advisory(result):
    if result["intent"] == "GREETING":
        print("Intent:", result["intent"])
        print(result["message"])
        return

    if result["intent"] == "INFORMATION":
        print("Intent:", result["intent"])
        print(result["message"])
        print("\nRetrieved Knowledge:")
        for i, chunk in enumerate(result["retrieved_knowledge"], 1):
            print(f"{i}. {chunk}")
        return

    if result["intent"] == "WATER_QUALITY":
        print("Intent:", result["intent"])
        print(result["message"])
        print("\nWater Quality Notes:")
        for n in result["water_quality_notes"]:
            print("-", n)
        print("\nRetrieved Knowledge:")
        for i, chunk in enumerate(result["retrieved_knowledge"], 1):
            print(f"{i}. {chunk}")
        return

    print("Intent:", result["intent"])
    print("\nPrimary Diagnosis:", result["primary_diagnosis"], f"({result['primary_confidence']:.4f})")

    print("\nTop 3 Differentials:")
    for i, (cls, score) in enumerate(result["top_3_differentials"], 1):
        print(f"{i}. {cls} ({score:.4f})")

    print("\nSensor Interpretation:")
    for note in result["sensor_interpretation"]:
        print("-", note)

    print("\nRetrieved Knowledge:")
    for i, chunk in enumerate(result["retrieved_knowledge"], 1):
        print(f"{i}. {chunk}")

    print("\nRecommended Actions:")
    for action in result["recommended_actions"]:
        print("-", action)

In [ ]:
queries = [
    "hi there",
    "my fish are gasping at the surface and breathing fast",
    "my fish has cotton like growth on the skin",
    "tell me about Aeromoniasis treatment",
    "check water quality because ammonia and oxygen seem bad"
]

for i, q in enumerate(queries):
    print("=" * 80)
    print("Query:", q)
    result, latency = run_aquasense_agent(q, sample_idx=i)
    print_advisory(result)
    print("\nLatency (ms):", {k: round(v, 2) for k, v in latency.items()})
    print()

Query: hi there
Intent: GREETING
Hello. I can help with fish disease diagnosis, water quality interpretation, and aquaculture information.

Latency (ms): {'intent_routing_ms': 0.02, 'end_to_end_ms': 0.02}

Query: my fish are gasping at the surface and breathing fast
Intent: GREETING
Hello. I can help with fish disease diagnosis, water quality interpretation, and aquaculture information.

Latency (ms): {'intent_routing_ms': 0.01, 'end_to_end_ms': 0.01}

Query: my fish has cotton like growth on the skin
Intent: INFORMATION
Information retrieval completed.

Retrieved Knowledge:
1. Cotton-like fungal growth on skin or fins is commonly associated with saprolegniasis.
2. Fungal infections such as saprolegniasis are more likely in stressed fish and damaged skin tissue.
3. Case record: repeated flashing behavior, rubbing on tank wall, skin irritation, and parasite presence under microscopy.

Latency (ms): {'intent_routing_ms': 0.02, 'rag_retrieval_ms': 52.34, 'end_to_end_ms': 52.36}

Query: te

In [ ]:
def build_structured_prompt(user_query, advisory_result):
    if advisory_result["intent"] != "DIAGNOSIS":
        return f"User Query: {user_query}\nIntent: {advisory_result['intent']}\n"

    prompt = f"""
You are an aquaculture advisory assistant.

User query:
{user_query}

Predicted primary diagnosis:
{advisory_result['primary_diagnosis']} with confidence {advisory_result['primary_confidence']:.4f}

Top differentials:
{advisory_result['top_3_differentials']}

Sensor interpretation:
{advisory_result['sensor_interpretation']}

Retrieved knowledge:
{advisory_result['retrieved_knowledge']}

Recommended actions:
{advisory_result['recommended_actions']}

Generate a concise, practical advisory for the fish farmer.
""".strip()

    return prompt

sample_result, _ = run_aquasense_agent("my fish are gasping at the surface and breathing fast", sample_idx=1)
sample_prompt = build_structured_prompt("my fish are gasping at the surface and breathing fast", sample_result)
print(sample_prompt)

User Query: my fish are gasping at the surface and breathing fast
Intent: GREETING



In [ ]:
benchmark_queries = [
    "my fish are gasping at the surface and breathing fast",
    "fish have red spots and skin lesions",
    "fish rubbing against tank wall and irritated",
    "white tail region visible and weak swimming",
    "cotton like growth on fins and skin"
]

latency_rows = []

for trial in range(100):
    q = random.choice(benchmark_queries)
    result, latency = run_aquasense_agent(q, sample_idx=trial)
    latency_rows.append(latency)

latency_df = pd.DataFrame(latency_rows)
display(latency_df.head())

,intent_routing_ms,end_to_end_ms,sensor_processing_ms,rag_retrieval_ms,text_inference_ms,sensor_inference_ms,cnn_inference_ms,fusion_ms,advisory_generation_ms
0,0.012425,0.012425,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.013162,0.013162,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.018211,36.249021,0.08345,36.147360,NaN,NaN,NaN,NaN,NaN
3,0.019363,29.617163,NaN,29.211836,0.033385,0.063979,0.038160,0.011903,0.238537
4,0.019480,76.060298,NaN,75.717359,0.027401,0.046535,0.034413,0.008784,0.206326


In [ ]:
summary_rows = []

for col in latency_df.columns:
    summary_rows.append({
        "component": col,
        "mean_ms": latency_df[col].mean(),
        "p50_ms": latency_df[col].quantile(0.50),
        "p95_ms": latency_df[col].quantile(0.95)
    })

latency_summary_df = pd.DataFrame(summary_rows).sort_values("component").reset_index(drop=True)
display(latency_summary_df)

,component,mean_ms,p50_ms,p95_ms
0,advisory_generation_ms,0.448546,0.193586,1.872041
1,cnn_inference_ms,0.064253,0.033557,0.059453
2,end_to_end_ms,22.014578,19.306738,58.286535
3,fusion_ms,0.008214,0.008049,0.011518
4,intent_routing_ms,0.021811,0.015826,0.025724
5,rag_retrieval_ms,35.192115,32.326048,75.243151
6,sensor_inference_ms,0.052317,0.053406,0.080987
7,sensor_processing_ms,0.134324,0.067953,0.725627
8,text_inference_ms,0.027104,0.027431,0.037552


In [ ]:
latency_trials_path = LATENCY_OUTPUT_DIR / "latency_trials.csv"
latency_summary_path = LATENCY_OUTPUT_DIR / "latency_summary.csv"
sample_prompt_path = LATENCY_OUTPUT_DIR / "sample_structured_prompt.txt"

latency_df.to_csv(latency_trials_path, index=False)
latency_summary_df.to_csv(latency_summary_path, index=False)

with open(sample_prompt_path, "w", encoding="utf-8") as f:
    f.write(sample_prompt)

print("Saved files:")
print(latency_trials_path)
print(latency_summary_path)
print(sample_prompt_path)

Saved files:
/content/drive/MyDrive/aquasense/notebook7_outputs/latency_trials.csv
/content/drive/MyDrive/aquasense/notebook7_outputs/latency_summary.csv
/content/drive/MyDrive/aquasense/notebook7_outputs/sample_structured_prompt.txt


In [ ]:
final_query = "my fish has ulcers, red patches, and is breathing fast near the surface"
final_result, final_latency = run_aquasense_agent(final_query, sample_idx=7)

print_advisory(final_result)
print("\nLatency (ms):", {k: round(v, 2) for k, v in final_latency.items()})

Intent: GREETING
Hello. I can help with fish disease diagnosis, water quality interpretation, and aquaculture information.

Latency (ms): {'intent_routing_ms': 0.02, 'end_to_end_ms': 0.02}


In [ ]:
print("AquaSense-Agent end-to-end prototype completed.")

print("\nModules connected in this notebook:")
print("- Intent Router")
print("- Text pathway probability generation")
print("- Sensor pathway probability loading")
print("- CNN placeholder probability loading")
print("- Late Bayesian fusion")
print("- Hybrid BM25 + FAISS retrieval")
print("- Advisory generation")
print("- Latency measurement")

AquaSense-Agent end-to-end prototype completed.

Modules connected in this notebook:
- Intent Router
- Text pathway probability generation
- Sensor pathway probability loading
- CNN placeholder probability loading
- Late Bayesian fusion
- Hybrid BM25 + FAISS retrieval
- Advisory generation
- Latency measurement
